### Structured Output

Structured output means making the AI return information in a fixed, predictable format instead of free-form text.

Example without structured output -

You ask: "Tell me about this product."

AI responds: "This laptop costs around $1200. It has 16 GB RAM and a 512 GB SSD."

This is easy for a person to read, but harder for another program to use because the wording can vary.

With structured output

You ask the AI to respond in JSON:

```json
{
  "name": "MacBook Air",
  "price": 1200,
  "ram": "16 GB",
  "storage": "512 GB SSD"
}

### Pydantic

Provides the richest feature set with field validation. description and nested structure


In [22]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000230C6311590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000230C63F8B90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    genre: str = Field(..., description="The genre of the movie")
    rating: float = Field(..., description="The rating of the movie (0-10)")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000230BE6BEC90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000230BE81A810>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [7]:
model_with_structure.invoke("Please provide details for the movie 'Inception' released in 2010, directed by Christopher Nolan, genre 'Science Fiction', with a rating of 8.8.")

Movie(title='Inception', year=2010, director='Christopher Nolan', genre='Science Fiction', rating=8.8)

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details for the movie 'Avengers'")
response


MovieDetails(title='The Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Iron Man'), Actor(name='Chris Evans', role='Captain America'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Mark Ruffalo', role='Hulk'), Actor(name='Scarlett Johansson', role='Black Widow'), Actor(name='Jeremy Renner', role='Hawkeye')], genres=['Action', 'Sci-Fi', 'Superhero'], budget=220.0)

### TypedDict

Typeddict provides similar functionality to Pydantic models but is more lightweight and doesn't require additional dependencies. It is part of the standard library in Python 3.8 and later.



In [10]:
from typing_extensions import TypedDict, Annotated

class MovieDetailsTypedDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float | None, ..., "The rating of the movie (0-10)"]

model_withtyped_dict = model.with_structured_output(MovieDetailsTypedDict)
response = model_withtyped_dict.invoke("Provide details for the movie 'The Matrix'")
response

{'director': 'Lana Wachowski, Lilly Wachowski',
 'rating': 8.7,
 'title': 'The Matrix',
 'year': 1999}

In [14]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    """An actor in a movie."""
    name: str
    role: str

class MovieDetails(TypedDict):
    """A movie with details."""
    title: str
    cast: list[Actor]
    genres: list[str]
    rating: float | None = Field(default=None, description="The rating of the movie (0-10)")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details for the movie 'The Matrix'")
response

{'cast': [{'name': 'Keanu Reeves', 'role': 'Neo'},
  {'name': 'Laurence Fishburne', 'role': 'Morpheus'},
  {'name': 'Carrie-Anne Moss', 'role': 'Trinity'},
  {'name': 'Hugo Weaving', 'role': 'Agent Smith'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'rating': 8.7,
 'title': 'The Matrix'}

### DataClasses

A data class is a class that is primarily used to store data and has minimal behavior. It is typically defined using the `@dataclass` decorator in Python, which automatically generates special methods like `__init__`, `__repr__`, and `__eq__` based on the class attributes. Data classes are useful for creating simple classes that hold data without having to write boilerplate code for initialization and representation.

In [15]:
import os
os.environ["OpenAI_API_KEY"] = os.environ.get("OpenAI_API_KEY")

In [25]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """A person's contact information."""
    name: str = Field(..., description="The name of the person")
    email: str = Field(..., description="The email address of the person")
    phone: str | None = Field(default=None, description="The phone number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3-32b", 
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the contact information from the following text: 'John Doe, email: zRw0i@example.com, phone: 123-456-7890.'"}]
})

print(result["structured_response"])

name='John Doe' email='zRw0i@example.com' phone='123-456-7890'


In [26]:
## TypedDict Example

from typing_extensions import TypedDict, Annotated

class ContactInfo(TypedDict):
    """A person's contact information."""
    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str | None, ..., "The phone number of the person"]

agent = create_agent(
    model = "groq:qwen/qwen3-32b", 
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the contact information from the following text: 'John Doe, email: zRw0i@example.com, phone: 123-456-7890.'"}]
})

print(result["structured_response"])

{'name': 'John Doe', 'email': 'zRw0i@example.com', 'phone': '123-456-7890'}


In [27]:
## Dataclass Example

from dataclasses import dataclass, field
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """A person's contact information."""
    name: str # The name of the person"
    email: str # The email address of the person"
    phone: str | None # The phone number of the person"


agent = create_agent(
    model = "groq:qwen/qwen3-32b", 
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the contact information from the following text: 'John Doe, email: zRw0i@example.com, phone: 123-456-7890.'"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='zRw0i@example.com', phone='123-456-7890')